# 01 — Data Cleaning
**Finding Hidden Gems: 25 Years of WNBA Draft Value and Player Success**

This notebook takes the raw WNBA draft dataset (1997–2022, 1,064 picks) and produces
an analysis-ready dataset: `data/wnba_draft_clean.csv`.

Steps:
1. Load and inspect the raw data
2. Fix known data gaps (verified against public draft records)
3. Standardize franchise identity across relocations/renames
4. Engineer draft-structure features (era, round, lottery status)
5. Engineer player-outcome features (never played, career length, per-season production)
6. Save the cleaned dataset + a data dictionary


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

raw = pd.read_csv('../data/wnba_draft_raw.csv')
print(raw.shape)
raw.head()


(1064, 14)


,overall_pick,year,team,player,former,college,years_played,games,win_shares,win_shares_40,minutes_played,points,total_rebounds,assists
0,1,2022,Atlanta Dream,Rhyne Howard,NaN,Kentucky,1,34.0,2.9,0.110,31.4,16.2,4.5,2.8
1,2,2022,Indiana Fever,NaLyssa Smith,NaN,Baylor,1,32.0,0.0,-0.001,30.7,13.5,7.9,1.4
2,3,2022,Washington Mystics,Shakira Austin,NaN,Ole Miss,1,36.0,3.1,0.160,21.6,8.7,6.5,0.9
3,4,2022,Indiana Fever,Emily Engstler,NaN,Louisville,1,35.0,0.4,0.024,18.2,5.2,5.2,1.5
4,5,2022,New York Liberty,Nyara Sabally,NaN,Oregon,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Initial inspection

In [2]:
print("Years covered:", raw.year.min(), "-", raw.year.max())
print("Total picks:", len(raw))
print("Unique teams (raw):", raw.team.nunique())
print()
print("Missing values:")
raw.isna().sum().sort_values(ascending=False)


Years covered: 1997 - 2022
Total picks: 1064
Unique teams (raw): 24

Missing values:


former            939
win_shares_40     335
minutes_played    334
win_shares        334
games             334
total_rebounds    334
assists           334
points            334
college            86
player              2
team                0
year                0
overall_pick        0
years_played        0
dtype: int64

In [3]:
# Picks per draft year varies (4-round drafts through 2002, then mostly 3 rounds).
# We'll use this to derive round numbers rather than assuming a fixed round size.
picks_per_year = raw.groupby('year')['overall_pick'].max()
teams_per_year = raw.groupby('year')['team'].nunique()
pd.DataFrame({'teams_with_picks': teams_per_year, 'total_picks': picks_per_year})


,teams_with_picks,total_picks
year,,
1997,8,32
1998,10,40
1999,12,50
2000,16,64
2001,16,64
2002,16,64
2003,14,42
2004,13,38
2005,13,39


## 2. Fix known data gaps

Three players are missing both `college` and `former` (international club) — these are
not international players, they're simply missing college data in the source. All three
are identifiable, well-documented WNBA draftees; values below are verified against
Basketball-Reference and WNBA.com official draft records:

- **Jenni Lingor** (2005, Pick 33, Detroit Shock) → Southwest Missouri State (Missouri State)
- **Jackie Stiles** (2001, Pick 4, Portland Fire) → Southwest Missouri State (Missouri State)
- **Tara Mitchem** (2001, Pick 60, New York Liberty) → Southwest Missouri State (Missouri State)

The dataset already spells this school "Missouri State" elsewhere, so we match that convention.

Two rows are also missing the `player` name — we leave the name blank but keep the row,
since pick/team/outcome data is intact and still valid for pick-value analysis.


In [4]:
df = raw.copy()

college_fixes = {
    ('Detroit Shock', 2005, 33): 'Missouri State',
    ('Portland Fire', 2001, 4): 'Missouri State',
    ('New York Liberty', 2001, 60): 'Missouri State',
}

for (team, year, pick), college in college_fixes.items():
    mask = (df.team == team) & (df.year == year) & (df.overall_pick == pick)
    assert mask.sum() == 1, f"expected 1 row for {team} {year} pick {pick}, got {mask.sum()}"
    df.loc[mask, 'college'] = college

print("Remaining missing college:", df.college.isna().sum(), "(all confirmed international picks)")
print("Remaining missing player name:", df.player.isna().sum())


Remaining missing college: 83 (all confirmed international picks)
Remaining missing player name: 2


## 3. Standardize franchise identity

Several WNBA franchises relocated and/or renamed over the dataset's 25 years. For pick-level
analysis the `team` column (team-at-the-time) is exactly what we want to keep. But for
**Phase 5 (team drafting ability)**, we need a `franchise` column that tracks a single
front office lineage across relocations, so a franchise's draft history isn't artificially
split across three names.

Also standardized: the two spellings of the San Antonio franchise ("San Antonio Silver
Stars" / "San Antonio Stars") are merged into `team` itself, since those are literally the
same team in the same city — not a relocation, just a mid-history rebrand.


In [5]:
# Same-city rebrand -> merge directly into `team`
df['team'] = df['team'].replace({'San Antonio Silver Stars': 'San Antonio Stars'})

# Cross-city relocations -> tracked separately as `franchise` (front-office lineage)
franchise_lineage = {
    'Utah Starzz': 'Las Vegas Aces',
    'San Antonio Stars': 'Las Vegas Aces',
    'Las Vegas Aces': 'Las Vegas Aces',

    'Orlando Miracle': 'Connecticut Sun',
    'Connecticut Sun': 'Connecticut Sun',

    'Detroit Shock': 'Dallas Wings',
    'Tulsa Shock': 'Dallas Wings',
    'Dallas Wings': 'Dallas Wings',
}
df['franchise'] = df['team'].map(franchise_lineage).fillna(df['team'])

# Franchises that folded outright (no relocation) - flag as defunct
defunct_teams = {'Charlotte Sting', 'Cleveland Rockers', 'Houston Comets',
                  'Miami Sol', 'Portland Fire', 'Sacramento Monarchs'}
df['franchise_active_2022'] = ~df['franchise'].isin(defunct_teams)

print("Franchises (lineage-adjusted):", df['franchise'].nunique())
df.groupby('franchise')['team'].unique()


Franchises (lineage-adjusted): 18


franchise
Atlanta Dream                                           [Atlanta Dream]
Charlotte Sting                                       [Charlotte Sting]
Chicago Sky                                               [Chicago Sky]
Cleveland Rockers                                   [Cleveland Rockers]
Connecticut Sun                      [Connecticut Sun, Orlando Miracle]
Dallas Wings                 [Dallas Wings, Tulsa Shock, Detroit Shock]
Houston Comets                                         [Houston Comets]
Indiana Fever                                           [Indiana Fever]
Las Vegas Aces         [Las Vegas Aces, San Antonio Stars, Utah Starzz]
Los Angeles Sparks                                 [Los Angeles Sparks]
Miami Sol                                                   [Miami Sol]
Minnesota Lynx                                         [Minnesota Lynx]
New York Liberty                                     [New York Liberty]
Phoenix Mercury                                       

## 4. Draft-structure features

In [6]:
# --- Draft era ---
# Bucketed around real WNBA structural inflection points:
#  1997-2002: Founding/Expansion era (league grew 8 -> 16 teams, 4 rounds)
#  2003-2009: Contraction era (16 -> 13 teams as early franchises folded/moved, 3-4 rounds)
#  2010-2015: Stabilization era (steady at 12 teams, 3 rounds)
#  2016-2022: Modern era (12 teams, 3 rounds, current CBA/roster rules)
def draft_era(year):
    if year <= 2002:
        return '1997-2002: Founding Era'
    elif year <= 2009:
        return '2003-2009: Contraction Era'
    elif year <= 2015:
        return '2010-2015: Stabilization Era'
    else:
        return '2016-2022: Modern Era'

df['draft_era'] = df['year'].apply(draft_era)

# --- Round number (derived from actual picks-per-team each year, not assumed) ---
df = df.sort_values(['year', 'overall_pick']).reset_index(drop=True)
df['pick_rank_in_year'] = df.groupby('year').cumcount()
teams_per_year_map = df.groupby('year')['team'].nunique()
df['teams_that_year'] = df['year'].map(teams_per_year_map)
df['round'] = (df['pick_rank_in_year'] // df['teams_that_year']) + 1
df['pick_in_round'] = (df['pick_rank_in_year'] % df['teams_that_year']) + 1
df = df.drop(columns=['pick_rank_in_year'])

# --- Round flags (spec asked for these explicitly) ---
df['lottery_pick'] = df['overall_pick'] <= 4          # top-4 picks, standard WNBA lottery size
df['first_round']  = df['round'] == 1
df['second_round'] = df['round'] == 2
df['third_round']  = df['round'] == 3
df['fourth_round'] = df['round'] >= 4                  # only existed 1997-2002ish

df[['year','overall_pick','teams_that_year','round','pick_in_round','lottery_pick','draft_era']].head(10)


,year,overall_pick,teams_that_year,round,pick_in_round,lottery_pick,draft_era
0,1997,1,8,1,1,True,1997-2002: Founding Era
1,1997,2,8,1,2,True,1997-2002: Founding Era
2,1997,3,8,1,3,True,1997-2002: Founding Era
3,1997,4,8,1,4,True,1997-2002: Founding Era
4,1997,5,8,1,5,False,1997-2002: Founding Era
5,1997,6,8,1,6,False,1997-2002: Founding Era
6,1997,7,8,1,7,False,1997-2002: Founding Era
7,1997,8,8,1,8,False,1997-2002: Founding Era
8,1997,9,8,2,1,False,1997-2002: Founding Era
9,1997,10,8,2,2,False,1997-2002: Founding Era


In [7]:
# Sanity check: round distribution should track known WNBA draft-round history
pd.crosstab(df['draft_era'], df['round'])


round,1,2,3,4,5
draft_era,,,,,
1997-2002: Founding Era,78,78,78,78,2
2003-2009: Contraction Era,94,94,93,1,0
2010-2015: Stabilization Era,72,72,72,0,0
2016-2022: Modern Era,82,82,82,6,0


## 5. Player-outcome features

In [8]:
# --- International flag ---
# `former` is NOT a clean international marker on its own - it's also populated for US
# college players who played in the old ABL (American Basketball League, folded 1998)
# before entering the WNBA draft (e.g. Dawn Staley, Jennifer Azzi). The reliable signal
# for "international, no US college" is simply that `college` is null (after our fixes
# above, remaining nulls are all confirmed international picks).
df['is_international'] = df['college'].isna()
df['prior_pro_experience'] = df['former'].notna()  # ABL and/or overseas pro club pre-draft

# --- Never played in the WNBA ---
# Confirmed: games is NaN exactly when a player never appeared in a WNBA game
# (no games==0 rows exist - NaN is a clean "never played" signal, not a data error)
df['never_played'] = df['games'].isna()

# --- Career outcome fields, with never_played handled explicitly ---
# years_played is already 0 for many never_played rows in the raw data; keep as-is.
# Games/win_shares/etc. stay NaN for never_played players (not 0) so aggregate stats
# (e.g. "average win shares among those who played") aren't silently dragged down.

print(f"International picks (no US college): {df['is_international'].sum()} ({df['is_international'].mean():.1%})")
print(f"Picks with prior pro experience (ABL and/or overseas club): {df['prior_pro_experience'].sum()}")
print(f"Picks who never played a WNBA game: {df['never_played'].sum()} ({df['never_played'].mean():.1%})")
print()
print("Never-played rate by round:")
print(df.groupby('round')['never_played'].mean().round(3))


International picks (no US college): 83 (7.8%)
Picks with prior pro experience (ABL and/or overseas club): 125
Picks who never played a WNBA game: 334 (31.4%)

Never-played rate by round:
round
1    0.043
2    0.239
3    0.603
4    0.541
5    0.000
Name: never_played, dtype: float64


In [9]:
# --- Career length bucket (useful for EDA / modeling later) ---
def career_bucket(years):
    if years == 0:
        return 'Never played'
    elif years <= 2:
        return '1-2 seasons'
    elif years <= 5:
        return '3-5 seasons'
    elif years <= 9:
        return '6-9 seasons'
    else:
        return '10+ seasons'

df['career_bucket'] = df['years_played'].apply(career_bucket)
df['career_bucket'].value_counts()


career_bucket
Never played    333
1-2 seasons     309
3-5 seasons     198
6-9 seasons     136
10+ seasons      88
Name: count, dtype: int64

## 6. College name check

The spec calls for standardizing college names. We checked the full list of 169 unique
college values for near-duplicates (e.g. abbreviation vs. full name referring to the same
school under a different string). None were found — every distinct string is a genuinely
distinct school, so no merge is needed. We do add a lightly cleaned display column that
strips generic suffixes, useful for chart labels later.


In [10]:
def short_college(name):
    if pd.isna(name):
        return name
    for suffix in [' University', ' College']:
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name

df['college_display'] = df['college'].apply(short_college)
# Confirm the cleanup didn't accidentally create new collisions
before, after = df['college'].nunique(), df['college_display'].nunique()
print(f"Unique colleges: {before} raw -> {after} display (collisions merged: {before - after})")


Unique colleges: 170 raw -> 170 display (collisions merged: 0)


## 7. Final schema + save

In [11]:
final_cols = [
    'overall_pick', 'year', 'draft_era', 'round', 'pick_in_round',
    'lottery_pick', 'first_round', 'second_round', 'third_round', 'fourth_round',
    'team', 'franchise', 'franchise_active_2022',
    'player', 'college', 'college_display', 'former', 'is_international', 'prior_pro_experience',
    'years_played', 'never_played', 'career_bucket',
    'games', 'win_shares', 'win_shares_40', 'minutes_played',
    'points', 'total_rebounds', 'assists',
]
clean = df[final_cols].copy()
clean.to_csv('../data/wnba_draft_clean.csv', index=False)
print(f"Saved data/wnba_draft_clean.csv — {clean.shape[0]} rows x {clean.shape[1]} columns")
clean.head()


Saved data/wnba_draft_clean.csv — 1064 rows x 29 columns


,overall_pick,year,draft_era,round,pick_in_round,lottery_pick,first_round,second_round,third_round,fourth_round,team,franchise,franchise_active_2022,player,college,college_display,former,is_international,prior_pro_experience,years_played,never_played,career_bucket,games,win_shares,win_shares_40,minutes_played,points,total_rebounds,assists
0,1,1997,1997-2002: Founding Era,1,1,True,True,False,False,False,Houston Comets,Houston Comets,False,Tina Thompson,USC,USC,NaN,False,False,17,False,10+ seasons,496.0,60.7,0.151,32.4,15.1,6.2,1.6
1,2,1997,1997-2002: Founding Era,1,2,True,True,False,False,False,Sacramento Monarchs,Sacramento Monarchs,False,Pamela McGee,USC,USC,NaN,False,False,2,False,1-2 seasons,57.0,2.0,0.064,22.1,8.6,4.6,0.6
2,3,1997,1997-2002: Founding Era,1,3,True,True,False,False,False,Los Angeles Sparks,Los Angeles Sparks,True,Jamila Wideman,Stanford,Stanford,NaN,False,False,4,False,3-5 seasons,84.0,-1.3,-0.037,16.6,2.2,1.4,2.5
3,4,1997,1997-2002: Founding Era,1,4,True,True,False,False,False,Cleveland Rockers,Cleveland Rockers,False,Eva Nemcova,NaN,NaN,Bourges (France),True,True,5,False,3-5 seasons,111.0,10.4,0.122,30.6,11.7,3.5,1.9
4,5,1997,1997-2002: Founding Era,1,5,False,True,False,False,False,Utah Starzz,Las Vegas Aces,True,Tammi Reiss,Virginia,Virginia,NaN,False,False,2,False,1-2 seasons,50.0,-0.1,-0.002,26.2,7.2,2.3,2.7


In [12]:
dictionary = pd.DataFrame([
    ('overall_pick', 'Overall pick number in the draft (1-64)'),
    ('year', 'Draft year, 1997-2022'),
    ('draft_era', 'Structural era bucket (Founding/Contraction/Stabilization/Modern)'),
    ('round', 'Draft round, derived from picks-per-team that year (not assumed fixed)'),
    ('pick_in_round', 'Pick number within that round'),
    ('lottery_pick', 'True if overall_pick <= 4'),
    ('first_round / second_round / third_round / fourth_round', 'Round flags'),
    ('team', 'Team that made the pick, at-the-time name (rebrand-merged, not relocation-merged)'),
    ('franchise', 'Front-office lineage across relocations (e.g. Utah Starzz -> Las Vegas Aces)'),
    ('franchise_active_2022', 'False if the franchise folded outright (no relocation)'),
    ('player', 'Player name'),
    ('college', 'US college attended (NaN if international, with 3 verified corrections)'),
    ('college_display', 'College name with University/College suffix stripped for labels'),
    ('former', 'Pre-draft club/country/league - covers both international clubs and the old ABL'),
    ('is_international', 'True if college is null (no US college - confirmed international pick)'),
    ('prior_pro_experience', 'True if former is populated (ABL and/or overseas pro club before the draft)'),
    ('years_played', 'Seasons played in the WNBA'),
    ('never_played', 'True if the player never appeared in a WNBA game'),
    ('career_bucket', 'Binned years_played'),
    ('games', 'Career games played (NaN if never played)'),
    ('win_shares', 'Career win shares (NaN if never played)'),
    ('win_shares_40', 'Win shares per 40 minutes'),
    ('minutes_played', 'Career average minutes per game'),
    ('points', 'Career average points per game'),
    ('total_rebounds', 'Career average rebounds per game'),
    ('assists', 'Career average assists per game'),
], columns=['column', 'description'])
dictionary.to_csv('../data/data_dictionary.csv', index=False)
dictionary


,column,description
0,overall_pick,Overall pick number in the draft (1-64)
1,year,"Draft year, 1997-2022"
2,draft_era,Structural era bucket (Founding/Contraction/St...
3,round,"Draft round, derived from picks-per-team that ..."
4,pick_in_round,Pick number within that round
5,lottery_pick,True if overall_pick <= 4
6,first_round / second_round / third_round / fou...,Round flags
7,team,"Team that made the pick, at-the-time name (reb..."
8,franchise,Front-office lineage across relocations (e.g. ...
9,franchise_active_2022,False if the franchise folded outright (no rel...
